# Acadia Hello World

We'll look at how to interact with an `Acadia` object and carry out the steps needed to compile and deploy a program. First, we'll discuss the system architecture a bit.

In the Acadia control system, there are many subsystems which each individually accept some form of instruction to do things. This notebook is (or rather, should be!) running on the ARM processors integrated into the RFSoC die, referred to collectively as the Processing Subsystem (PS). However, the bulk of the system's specialized capabilities are enabled by custom logic circuits in the Programmable Logic region (PL). 

The first of the custom modules we'll explore is the Acadia sequencer, which is required for doing much of anything involving other parts of the hardware. 

First, import the necessary definitions:

In [1]:
from acadia.system import Acadia

No module named 'pyxrfdc'
No module named 'pyxrfclk'


We can then just instantiate an `Acadia` object and interact with its sequencer. In order to do anything, we'll need to establish a means of communication between the PS and the PL; for this we'll use the PS GPIO, which is a set of signals exposed to the PL from the PS. These signals can be configured as inputs (to the PS) or as outputs (from the PS); the PS can assign values to the output signals or read values from the input signals by writing or reading special-purpose registers in its address space. The sequencer can drive the inputs or read the outputs using modules attached to its bus.

As a simple example, we'll create a sequence in which the sequencer reads a value from GPIO port 3 into a register, adds 1 to it, and then writes it to GPIO port 4:

In [2]:
# We'll choose arbitrarily to use port 3
PS_TO_PL_GPIO_PORT = 3
PL_TO_PS_GPIO_PORT = 4

acadia = Acadia()

# Create a sequence for the sequencer to just add 1 forever to the GPIO value
def sequence(acda):
    with acda.sequencer() as seq:
        reg = seq.Register()
        
        with seq.loop():
            reg.load(acda.gpio_read(PS_TO_PL_GPIO_PORT))
            reg += 1
            acda.gpio_write(PL_TO_PS_GPIO_PORT, reg)
        

Then, we compile the program:

In [3]:
acadia.compile(sequence)

It's worthwhile to examine the compiled output and identify how it maps onto the commands:

In [4]:
acadia._sequencer_type.instances[0]._compiled_program

[STP(src1=IMM, src2=REG0, dest1=BUS_ADDR, dest2=REG0, imm1=1277952, imm2=0, dsp_cep=None, push_return=False, comment=None),
 STP(src1=REG0, src2=REG0, dest1=REG0, dest2=REG0, imm1=0, imm2=0, dsp_cep=None, push_return=False, comment='Pipeline latency for bus'),
 STP(src1=REG0, src2=REG0, dest1=REG0, dest2=REG0, imm1=0, imm2=0, dsp_cep=None, push_return=False, comment='Pipeline latency for bus'),
 STP(src1=REG0, src2=REG0, dest1=REG0, dest2=REG0, imm1=0, imm2=0, dsp_cep=None, push_return=False, comment='Pipeline latency for bus'),
 STP(src1=BUS_DATA, src2=REG0, dest1=REG0, dest2=REG0, imm1=0, imm2=0, dsp_cep=None, push_return=False, comment=None),
 STP(src1=REG0, src2=REG0, dest1=DSP_AB0, dest2=REG0, imm1=0, imm2=0, dsp_cep=None, push_return=False, comment=None),
 STP(src1=IMM, src2=IMM, dest1=DSP_CFG0, dest2=DSP_C0, imm1=DSPConfiguration(mode='AB+C', rst_p=False, dsp_cep='pulse'), imm2=1, dsp_cep=None, push_return=False, comment=None),
 STP(src1=REG0, src2=REG0, dest1=REG0, dest2=REG0, 

Now, we can attach to the hardware and load the program:

In [5]:
acadia.attach()
acadia.assemble(load=True)

Before we run the sequencer, let's show that nothing is currently happening on the GPIO ports. We first need to configure the direction of the ports:

In [6]:
acadia.gpio_set_direction(PS_TO_PL_GPIO_PORT, 0xFFFFFFFF)
acadia.gpio_set_direction(PL_TO_PS_GPIO_PORT, 0x00000000)

Then, let's write some arbitrary number to the output port:

In [7]:
acadia.gpio_write(PS_TO_PL_GPIO_PORT, 8)

We can check by reading back the value from the input port that the sequencer isn't actively driving the output:

In [8]:
acadia.gpio_read(PL_TO_PS_GPIO_PORT)

19

Now let's run the sequencer and check the output again:

In [9]:
acadia.sequencer_reset()
acadia.sequencer_run(sequence)

In [10]:
acadia.gpio_read(PL_TO_PS_GPIO_PORT)

9

As we programmed, the sequencer updated the GPIO input port with the value on the output port plus 1. Because this is occurring in an infinite loop, we can try with another value and interact with the sequencer while it runs:

In [11]:
acadia.gpio_write(PS_TO_PL_GPIO_PORT, 12)
acadia.gpio_read(PL_TO_PS_GPIO_PORT)

13

Now let's halt the sequencer's execution and show that the output stops updating:

In [12]:
acadia.sequencer_halt()
acadia.gpio_write(PS_TO_PL_GPIO_PORT, 20)
acadia.gpio_read(PL_TO_PS_GPIO_PORT)

13

Because the sequencer isn't running, the GPIO ports hold their previous values.